# Model Evaluation
**CubicNova Data Science Roadmap — Module 09**

This notebook demonstrates classification and regression evaluation, baselines, cross-validation, and learning curves. Run cells from top to bottom.

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris, load_diabetes, make_classification
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, learning_curve
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report, roc_auc_score,
    average_precision_score, mean_absolute_error, mean_squared_error, r2_score,
    ConfusionMatrixDisplay
)

## 2. Classification: baseline vs. trained model

In [ ]:
iris = load_iris()
X, y = iris.data, iris.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
models = {
    "Most-frequent baseline": DummyClassifier(strategy="most_frequent"),
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
}
rows = []
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    rows.append({"model": name, "accuracy": accuracy_score(y_test, pred),
                 "macro_f1": f1_score(y_test, pred, average="macro", zero_division=0)})
display(pd.DataFrame(rows).set_index("model").round(3))

## 3. Confusion matrix and class-wise report

In [ ]:
classifier = LogisticRegression(max_iter=1000)
classifier.fit(X_train, y_train)
pred = classifier.predict(X_test)
print("Confusion matrix:\n", confusion_matrix(y_test, pred))
print(classification_report(y_test, pred, target_names=iris.target_names, zero_division=0))
ConfusionMatrixDisplay.from_predictions(y_test, pred, display_labels=iris.target_names)
plt.title("Iris confusion matrix")
plt.tight_layout()
plt.show()

## 4. Cross-validation

Cross-validation helps compare models across multiple folds. Keep the final test set separate.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(LogisticRegression(max_iter=1000), X_train, y_train,
                         cv=cv, scoring="accuracy")
print("Fold scores:", scores)
print("Mean:", round(scores.mean(), 3))
print("Standard deviation:", round(scores.std(), 3))

## 5. Regression metrics and baseline comparison

In [ ]:
diabetes = load_diabetes()
Xr, yr = diabetes.data, diabetes.target
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    Xr, yr, test_size=0.2, random_state=42
)
rows = []
for name, model in {
    "Mean baseline": DummyRegressor(strategy="mean"),
    "Linear Regression": LinearRegression(),
}.items():
    model.fit(X_train_r, y_train_r)
    pred_r = model.predict(X_test_r)
    rows.append({"model": name, "MAE": mean_absolute_error(y_test_r, pred_r),
                 "RMSE": np.sqrt(mean_squared_error(y_test_r, pred_r)),
                 "R2": r2_score(y_test_r, pred_r)})
display(pd.DataFrame(rows).set_index("model").round(3))

## 6. Imbalanced binary classification

ROC-AUC and average precision evaluate ranking across thresholds. They do not by themselves select an operating threshold.

In [ ]:
Xb, yb = make_classification(n_samples=1000, n_features=20, n_informative=8,
                             n_redundant=4, weights=[0.85, 0.15], random_state=42)
X_train_b, X_test_b, y_train_b, y_test_b = train_test_split(
    Xb, yb, test_size=0.2, random_state=42, stratify=yb
)
binary_model = LogisticRegression(max_iter=1000, class_weight="balanced")
binary_model.fit(X_train_b, y_train_b)
pred_b = binary_model.predict(X_test_b)
proba_b = binary_model.predict_proba(X_test_b)[:, 1]
print("Accuracy:", round(accuracy_score(y_test_b, pred_b), 3))
print("Balanced accuracy:", round(balanced_accuracy_score(y_test_b, pred_b), 3))
print("Precision:", round(precision_score(y_test_b, pred_b, zero_division=0), 3))
print("Recall:", round(recall_score(y_test_b, pred_b, zero_division=0), 3))
print("F1:", round(f1_score(y_test_b, pred_b, zero_division=0), 3))
print("ROC-AUC:", round(roc_auc_score(y_test_b, proba_b), 3))
print("Average precision:", round(average_precision_score(y_test_b, proba_b), 3))
print("Confusion matrix:\n", confusion_matrix(y_test_b, pred_b))

## 7. Learning curve

Learning curves help diagnose overfitting and underfitting. This is a diagnostic illustration; in a real workflow, keep final test data separate.

In [ ]:
X_lc, y_lc = load_iris(return_X_y=True)
cv_lc = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
sizes, train_scores, valid_scores = learning_curve(
    DecisionTreeClassifier(random_state=42), X_lc, y_lc, cv=cv_lc,
    scoring="accuracy", train_sizes=np.linspace(0.2, 1.0, 5)
)
plt.figure(figsize=(6, 4))
plt.plot(sizes, train_scores.mean(axis=1), marker="o", label="Training")
plt.plot(sizes, valid_scores.mean(axis=1), marker="o", label="Cross-validation")
plt.xlabel("Training examples")
plt.ylabel("Accuracy")
plt.title("Learning curve: Decision Tree")
plt.legend()
plt.tight_layout()
plt.show()

## 8. Key Takeaways
- Match evaluation splits to the deployment scenario.
- Use a baseline and metrics suited to the task and error costs.
- Cross-validation helps model selection but does not fix leakage.
- Classification needs more than accuracy for imbalanced data.
- Regression metrics capture different aspects of error.
- Inspect errors and limitations; do not treat one score as a guarantee.

Complete `practice.md` and compare your work with `solution.md`.

Further reading: [Scikit-learn model evaluation](https://scikit-learn.org/stable/modules/model_evaluation.html), [cross-validation](https://scikit-learn.org/stable/modules/cross_validation.html).